# LaLonde Dataset: Propensity Score Matching with CausalML

This notebook demonstrates propensity score matching on the LaLonde dataset (National Supported Work demonstration program) to estimate the average treatment effect on the treated (ATT) of job training on earnings.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from causalml.match import NearestNeighborMatch
from dowhy import datasets

In [ ]:
# Load LaLonde dataset from DoWhy
data = datasets.lalonde_dataset()

print(f"Dataset shape: {data.shape}")
print(f"\nColumns: {list(data.columns)}")
print(f"\nFirst few rows:")
data.head()

In [ ]:
# Define treatment, outcome, and covariates
treatment_col = "treat"
outcome_col = "re78"
covariate_cols = ["age", "educ", "black", "hisp", "married", "nodegr", "re74", "re75"]

# Fit propensity scores using logistic regression
X = data[covariate_cols]
treatment = data[treatment_col]

psmodel = LogisticRegression(max_iter=2000, random_state=42)
psmodel.fit(X, treatment)
data["propensity_score"] = psmodel.predict_proba(X)[:, 1]

# Perform nearest neighbor matching on propensity scores
matcher = NearestNeighborMatch(caliper=0.2, replace=False)
matched_data = matcher.match(data=data, treatment_col=treatment_col, score_cols=["propensity_score"])

print(f"Original data: {len(data)} rows")
print(f"Matched data: {len(matched_data)} rows")

In [ ]:
# Compute Average Treatment Effect on the Treated (ATT)
treated_outcome = matched_data.loc[matched_data[treatment_col] == 1, outcome_col].mean()
control_outcome = matched_data.loc[matched_data[treatment_col] == 0, outcome_col].mean()
att = treated_outcome - control_outcome

print(f"\nAverage outcome for treated: ${treated_outcome:,.2f}")
print(f"Average outcome for control: ${control_outcome:,.2f}")
print(f"\nEstimated ATT: ${att:,.2f}")